## BREADTH-FIRST-SEARCH

function BREADTH-FIRST-SEARCH(problem) returns a solution or failure  
    node ← NODE(problem.INITIAL)  
    if problem.GOAL-TEST(node.STATE) then  
        return SOLUTION(node)  

    frontier ← FIFO-QUEUE()    # danh sách node chờ mở rộng  
    frontier.INSERT(node)  

    reached ← ∅    # tập các state đã khám phá  

    while not EMPTY?(frontier) do  
        node ← frontier.REMOVE()    # lấy node đầu tiên trong queue  
        reached ← reached ∪ {node.STATE}  

        if problem.GOAL-TEST(node.STATE) then  
            return SOLUTION(node)  

        for each action in problem.ACTIONS(node.STATE) do  
            child ← CHILD_NODE(problem, node, action)  

            if child.STATE ∉ reached ∧ child ∉ frontier then  
                frontier.INSERT(child)  

    return failure

In [ ]:
import random
import sys

sys.stdout.reconfigure(encoding='utf-8')

from collections import deque

class Node(object):
    def __init__(self, state, parent, action, level):
        self.state = state
        self.parent = parent
        self.action = action
        self.level = level

class QueueFrontier():
    def __init__(self):
        self.frontier = deque()
        self.frontier_states = set()
    
    def add(self, node):
        self.frontier.append(node)
        self.frontier_states.add(node.state)

    def contains_state(self, state):
        return state in self.frontier_states

    def empty(self):
        return len(self.frontier) == 0
    
    def remove(self):
        if self.empty():
            raise Exception("empty frontier")
        node = self.frontier.popleft()
        self.frontier_states.remove(node.state)
        return node



GOAL_STATE = ((1,2,3),
              (4,5,6),
              (7,8,0))
    
def possible_actions(state):
    """
    Tính toán các bước có thể đi của 0
    """
    moves = []
    zero_row, zero_col = find_zero(state)
    
    if zero_col - 1 >= 0:
        moves.append("LEFT")
    if zero_col + 1 < 3:
        moves.append("RIGHT")
    if zero_row - 1 >= 0:
        moves.append("UP")
    if zero_row + 1 < 3:
        moves.append("DOWN")
    return moves

def find_zero(state):
    for i in range(3):
        for j in range(3):
            if state[i][j] == 0:
                return i, j

def matrix_to_tuple(matrix):
    """
    Đổi ma trận sang tuple
    """
    return tuple(tuple(row) for row in matrix)

def generate_random_board():
    """
    Tạo ra một bàn cờ ngẫu nhiên
    """
    numbers = list(range(9))
    
    random.shuffle(numbers)
    
    board = [numbers[i:i+3] for i in range(0, 9, 3)]
    
    return matrix_to_tuple(board)
    
def apply_move(board, action):
    """
    Áp dụng các bước
    """
    r, c = find_zero(board)
    new_board = [list(row) for row in board]
    if action == "UP":
        new_board[r][c], new_board[r-1][c] = new_board[r-1][c], new_board[r][c]
    elif action == "DOWN":
        new_board[r][c], new_board[r+1][c] = new_board[r+1][c], new_board[r][c]
    elif action == "LEFT":
        new_board[r][c], new_board[r][c-1] = new_board[r][c-1], new_board[r][c]
    elif action == "RIGHT":
        new_board[r][c], new_board[r][c+1] = new_board[r][c+1], new_board[r][c]
    return matrix_to_tuple(new_board)

    
def main():
    board = generate_random_board()
    frontier = QueueFrontier()
    reached = set()
    step_counting = 0
    
    n1 = Node(board, None, None, 0)
    frontier.add(n1)

    while True:
        if frontier.empty():
            print(f"Không tìm thấy lời giải, đã thực hiện {step_counting} bước!")
            break
        
        current_node = frontier.remove()
        current_state = current_node.state
        step_counting += 1

        if current_state == GOAL_STATE:
            print(f"Đã tìm thấy lời giải sau {step_counting} lần duyệt nhánh!\n")
            
            # Truy vết ngược để lấy đường đi
            path = []
            node = current_node
            while node is not None:
                path.append(node)
                node = node.parent
            path.reverse()
            
            print(f"Số bước di chuyển thực tế để đến đích: {len(path) - 1}")
            print("Đường đi chi tiết:")
            for step, n in enumerate(path):
                action_str = f"Hành động: {n.action}" if n.action else "Trạng thái ban đầu"
                print(f"Bước {step} ({action_str}):")
                for row in n.state:
                    print(row)
                print()
            return 
            
        if current_state not in reached:
            reached.add(current_state)
        
        valid_moves = possible_actions(current_state)

        for action in valid_moves:
            new_state = apply_move(current_state, action)
            if not frontier.contains_state(new_state) and new_state not in reached:
                frontier.add(Node(new_state, current_node, action, current_node.level + 1))
        
if __name__ == "__main__":
    main()
    